In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import randint

# Load the data
df = pd.read_excel(r"C:\Users\91826\OneDrive\Desktop\Kisaan_vaani\Kisaan-vaani\Datasets\Crop_predicition_dataset.xlsx")

# Split the data into features and target
x = df[['N', 'P', 'K', 'Temperature', 'Humidity', 'ph', 'Rainfall']]
y = df['Label']

# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x.values, y, test_size=0.25, random_state=0)

# Initialize the classifier
rf = RandomForestClassifier(random_state=42)

# Define the parameter grid
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 11),
    'max_features': [None,'sqrt', 'log2']
}

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter= 50, cv=5, random_state=42, n_jobs=-1)

# Fit the model
random_search.fit(x_train, y_train)

# Get the best model
best_rf = random_search.best_estimator_

# Predict and calculate accuracy
y_pred = best_rf.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy after hyperparameter tuning:", accuracy)

Accuracy after hyperparameter tuning: 0.9730526315789474


In [12]:
# Predict the top N crops for a new sample
new_sample = np.array([98, 32, 60.2, 26.42, 65.77, 6.28, 101.93]).reshape(1, -1)
probs = best_rf.predict_proba(new_sample)
N = 5
top_n_idx = np.argsort(probs[0])[-N:][::-1]
top_n_labels = best_rf.classes_[top_n_idx]

print("Top", N, "crops for the given sample:", top_n_labels)

Top 5 crops for the given sample: ['Rose' 'maize' 'coffee' 'Bottle Gourd' 'Jute']


In [13]:
import pandas as pd

def get_common_name_for_label(input_file, desired_labels):
    
    df = pd.read_excel(input_file)
    common_names_dict = {}

    # Iterate over the input list of labels
    for label in desired_labels:
        # Find the corresponding common name for the label
        common_name = df.loc[df['Label'] == label, 'Common Names'].values
        # If a common name is found, add it to the dictionary
        if len(common_name) > 0:
            common_names_dict[label] = common_name[0]

    # Generate the output list of common names maintaining the order of input labels
    common_names = [common_names_dict[label] for label in desired_labels if label in common_names_dict]

    return common_names

# Example usage
input_file = r"C:\Users\91826\OneDrive\Desktop\Kisaan_vaani\Kisaan-vaani\Datasets\Crop_common_name.xlsx"  # Replace 'data.xlsx' with your file path
desired_labels =  top_n_labels # Specify the desired labels
common_names = get_common_name_for_label(input_file, desired_labels)
print(common_names)


['Gulab', 'Makka', 'coffee', 'laukee', 'Jute']
